# 02. Generación de pseudo-LiDAR para varios samples

Este notebook corresponde al segundo paso del experimento.

En esta etapa:
- se toma el manifest definido en el paso 1,
- se genera una pseudo-LiDAR para cada sample incluido en dicho manifest,
- y se guarda un resultado independiente por frame.

Justificación:
- para estudiar una aplicación orientada a SLAM no es suficiente con un único frame,
- sino que es necesario disponer de una pseudo-LiDAR por timestamp para poder comparar `t` con `t+1`.


In [1]:
from pathlib import Path
import json
import subprocess


In [2]:
ROOT = Path('/home/clara/ml-depth-pro/slam_readiness_nuscenes')
MANIFEST_PATH = ROOT / 'manifests' / 'scene-0061_first5.json'
SCRIPT_PATH = ROOT / 'scripts' / 'generate_pseudolidar_manifest.py'
OUTPUT_ROOT = ROOT / 'outputs'

manifest = json.loads(MANIFEST_PATH.read_text())
manifest

{'version': 'v1.0-mini',
 'dataroot': '/home/clara/datasets/nuscenes',
 'scene_name': 'scene-0061',
 'scene_token': 'cc8c0bf57f984915a77078b10eb33198',
 'description': 'Parked truck, construction, intersection, turn left, following a van',
 'num_requested': 5,
 'num_selected': 5,
 'samples': [{'index': 0,
   'sample_token': 'ca9a282c9e77460f8360f564131a8af5',
   'timestamp_s': 1532402927.647951,
   'prev': '',
   'next': '39586f9d59004284a7114a68825e8eec'},
  {'index': 1,
   'sample_token': '39586f9d59004284a7114a68825e8eec',
   'timestamp_s': 1532402928.147847,
   'prev': 'ca9a282c9e77460f8360f564131a8af5',
   'next': '356d81f38dd9473ba590f39e266f54e5'},
  {'index': 2,
   'sample_token': '356d81f38dd9473ba590f39e266f54e5',
   'timestamp_s': 1532402928.698048,
   'prev': '39586f9d59004284a7114a68825e8eec',
   'next': 'e0845f5322254dafadbbed75aaa07969'},
  {'index': 3,
   'sample_token': 'e0845f5322254dafadbbed75aaa07969',
   'timestamp_s': 1532402929.197353,
   'prev': '356d81f38dd9473

## Función del script

Para cada sample incluido en el manifest, el script realiza las siguientes operaciones:
1. carga las 6 cámaras de `nuScenes`,
2. estima la profundidad de cada imagen mediante `Depth Pro`,
3. reconstruye la nube 3D fusionada (`ring`),
4. la convierte a una representación `pseudo-LiDAR`,
5. y guarda un archivo `.ply` por sample.

En esta etapa todavía **no se evalúa SLAM**. El objetivo es preparar la secuencia de nubes que se utilizará en los análisis posteriores.


## Nota sobre `max-samples`

Para que los notebooks posteriores puedan evaluar registro temporal, deben existir al menos dos samples procesados. En este experimento se usan cinco samples, por lo que `--max-samples` debe mantenerse en `5` para reproducir los resultados finales.

Si se cambia temporalmente a `1` para una prueba r?pida, el fichero `outputs/scene-0061/run_summary.json` queda sobrescrito con un ?nico frame. En ese caso, el notebook `03_pairwise_registration.ipynb` devolver? `num_pairs = 0` porque no habr? parejas consecutivas que registrar.


In [3]:
cmd = [
    'python',
    str(SCRIPT_PATH),
    '--manifest', str(MANIFEST_PATH),
    '--output-root', str(OUTPUT_ROOT),
    '--max-samples', '5',
]
print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print('returncode:', result.returncode)


python /home/clara/ml-depth-pro/slam_readiness_nuscenes/scripts/generate_pseudolidar_manifest.py --manifest /home/clara/ml-depth-pro/slam_readiness_nuscenes/manifests/scene-0061_first5.json --output-root /home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs --max-samples 5
{
  "sample_token": "ca9a282c9e77460f8360f564131a8af5",
  "index": 0,
  "timestamp_s": 1532402927.647951,
  "ring_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_ring_6cams_ego.ply",
  "pseudo_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_pseudolidar_ego.ply",
  "lidar_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_lidar_top_ego.ply",
  "ring_num_points": 281010,
  "pseudo_num_points": 23534
}
{
  "sample_token": "39586f9d59004284a7114a68825e8eec",
  "index": 1,
  "timestamp_s": 1532402928.147847,
  "ring_path": "/home/

In [4]:
scene_dir = OUTPUT_ROOT / manifest['scene_name']
summary_path = scene_dir / 'run_summary.json'
if summary_path.exists():
    print(summary_path)
    print(summary_path.read_text())
else:
    print('Todavia no existe run_summary.json. Ejecuta antes la celda del script.')


/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/run_summary.json
{
  "scene_name": "scene-0061",
  "manifest_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/manifests/scene-0061_first5.json",
  "num_processed": 5,
  "samples": [
    {
      "sample_token": "ca9a282c9e77460f8360f564131a8af5",
      "index": 0,
      "timestamp_s": 1532402927.647951,
      "ring_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_ring_6cams_ego.ply",
      "pseudo_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_pseudolidar_ego.ply",
      "lidar_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_lidar_top_ego.ply",
      "ring_num_points": 281010,
      "pseudo_num_points": 23534
    },
    {
      "sample_token": "39586f9d59004284a7114a68825e8eec",
      "index": 1,
      "timestamp_s": 153

## Resultado esperado al final de esta etapa

Dentro de `outputs/scene-0061/<sample_token>/` debería aparecer, para cada sample:
- `pcd_ring_6cams_ego.ply`,
- `pcd_pseudolidar_ego.ply`,
- `summary.json`.

Además, a nivel de escena se generará:
- `run_summary.json`.

Estos archivos constituirán la base para el paso 3, centrado en la comparación temporal entre nubes consecutivas.


## Visualizacion 3D de la generacion de pseudo-LiDAR

Ademas de las metricas y los ficheros `.ply`, se incluyen visualizaciones interactivas para inspeccionar la nube densa fusionada y la pseudo-LiDAR resultante. Esto permite ver el paso desde la reconstruccion 3D generada por las camaras hasta una representacion mas parecida a un LiDAR.

Enlaces directos:

- [Nube densa fusionada](../outputs/scene-0061/pointcloud_phase_visualizations/phase_01_dense_ring_6cams.html)
- [Pseudo-LiDAR frente a LiDAR real](../outputs/scene-0061/pointcloud_phase_visualizations/phase_02_pseudolidar_vs_lidar_gt.html)


In [5]:
from IPython.display import IFrame, display, Markdown
VIS_DIR = ROOT / 'outputs' / 'scene-0061' / 'pointcloud_phase_visualizations'
display(Markdown('[Abrir indice de visualizaciones 3D](' + str(VIS_DIR / 'index.html') + ')'))
display(IFrame(src=str(VIS_DIR / 'phase_01_dense_ring_6cams.html'), width='100%', height=720))
display(IFrame(src=str(VIS_DIR / 'phase_02_pseudolidar_vs_lidar_gt.html'), width='100%', height=720))


[Abrir indice de visualizaciones 3D](/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/pointcloud_phase_visualizations/index.html)